# Example 2.3 / 1: Lists for Sets, Dictionaries for Parameters

Companion notebook for **2.3 Data from Python Lists and Dictionaries** in the AMPL lab Documentation, section "Lists for Sets, Dictionaries for Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Lists for Sets, Dictionaries for Parameters

A dictionary can be the single
source of truth for both the set of keys and the indexed values.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # Products
param profit {PROD};       # Profit per ton
param market {PROD} >= 0;  # Maximum sales in tons

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
profit = {"bands": 25, "coils": 30, "plate": 29}
market = {"bands": 6000, "coils": 4000, "plate": 3500}

ampl.set["PROD"].set_values(profit.keys())
ampl.param["profit"].set_values(profit)
ampl.param["market"] = market

In [5]:
# run.py
print(ampl.param["profit"].to_dict())
print(ampl.param["market"]["coils"])

{'bands': 25, 'coils': 30, 'plate': 29}
4000


### Pure AMPL: data.dat and model.run

`to_dict()` returns every stored value; indexing the `Parameter` with one key returns that value
directly.

The equivalent pure-AMPL data is a key/value list:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display profit;
display market['coils'];

profit [*] :=
bands  25
coils  30
plate  29
;

market['coils'] = 4000

